# DX 704 Week 8 Project

This homework will modify a simulator controlling a small vehicle to implement tabular q-learning.
You will first test your code with random and greedy-epsilon policies, then tweak your own training method for a more optimal policy.

The full project description and a template notebook are available on GitHub: [Project 8 Materials](https://github.com/bu-cds-dx704/dx704-project-08).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Rover Simulator

The following Python class implements a simulation of a simple vehicle with integer x,y coordinates facing in one of 8 possible directions.


In [1]:
# DO NOT CHANGE

import random

class RoverSimulator(object):
    DIRECTIONS = ((0, 1), (1, 1), (1, 0), (1, -1), (0, -1), (-1, -1), (-1, 0), (-1, 1))

    def __init__(self, resolution):
        self.resolution = resolution
        self.terminal_state = self.construct_state(resolution // 2, resolution // 2, 0)

        self.initial_states = []
        for initial_x in (0, resolution // 2, resolution - 1):
            for initial_y in (0, resolution // 2, resolution - 1):
                for initial_direction in range(8):
                    initial_state = self.construct_state(initial_x, initial_y, initial_direction)
                    if initial_state != self.terminal_state:
                        self.initial_states.append(initial_state)

    def construct_state(self, x, y, direction):
        assert 0 <= x < self.resolution
        assert 0 <= y < self.resolution
        assert 0 <= direction < 8

        state = (y * self.resolution + x) * 8 + direction
        assert self.decode_state(state) == (x, y, direction)
        return state

    def decode_state(self, state):
        direction = state % 8
        x = (state // 8) % self.resolution
        y = state // (8 * self.resolution)

        return (x, y, direction)

    def get_actions(self, state):
        return [-1, 0, 1]

    def get_next_reward_state(self, curr_state, curr_action):
        if curr_state == self.terminal_state:
            # no rewards or changes from terminal state
            return (0, curr_state)

        (curr_x, curr_y, curr_direction) = self.decode_state(curr_state)
        (curr_dx, curr_dy) = self.DIRECTIONS[curr_direction]

        assert self.construct_state(curr_x, curr_y, curr_direction) == curr_state

        assert curr_action in (-1, 0, 1)

        next_x = min(max(0, curr_x + curr_dx), self.resolution - 1)
        next_y = min(max(0, curr_y + curr_dy), self.resolution - 1)
        next_direction = (curr_direction + curr_action) % 8

        next_state = self.construct_state(next_x, next_y, next_direction)
        next_reward = 1 if next_state == self.terminal_state else 0

        return (next_reward, next_state)

    def rollout_policy(self, policy_func, max_steps=1000):
        curr_state = self.sample_initial_state()
        for i in range(max_steps):
            curr_action = policy_func(curr_state, self.get_actions(curr_state))
            (next_reward, next_state) = self.get_next_reward_state(curr_state, curr_action)
            yield (curr_state, curr_action, next_reward, next_state)
            curr_state = next_state

    def sample_initial_state(self):
        return random.choice(self.initial_states)

In [2]:
simulator = RoverSimulator(16)
initial_sample = simulator.sample_initial_state()
print("INITIAL SAMPLE", initial_sample)

INITIAL SAMPLE 1146


## Part 1: Implement a Random Policy

Random policies are often used to test simulators and start initial exploration.
Implement a random policy for these simulators.

In [3]:
# YOUR CHANGES HERE

def random_policy(state, actions):
    return 0

Use the code below to test your random policy.
Then modify it to save the results in "log-random.tsv" with the columns curr_state, curr_action, next_reward and next_state.

In [5]:
import csv

def write_tsv(filename, columns, rows):
    with open(filename, "w", newline="") as file:
        writer = csv.writer(file, delimiter="\t")
        writer.writerow(columns)
        writer.writerows(rows)

random.seed(2026)
random_rows = list(simulator.rollout_policy(random_policy, max_steps=32))
write_tsv("log-random.tsv", ["curr_state", "curr_action", "next_reward", "next_state"], random_rows)
print("Saved", len(random_rows), "random transitions")


Saved 32 random transitions


Submit "log-random.tsv" in Gradescope.

## Part 2: Implement Q-Learning with Random Policy

The code below runs 32 random rollouts of 1024 steps using your random policy.
Modify the rollout code to implement Q-Learning.
Just implement one learning update for each sampled state-action in the simulation.
Use $\alpha=1$ and $\gamma=0.9$ since the simulator is deterministic and there is a sink where the rewards stop.




In [7]:
from collections import defaultdict

ALPHA = 1.0
GAMMA = 0.9
ACTIONS = (-1, 0, 1)

def make_q_table():
    return defaultdict(float)

def q_update(q, curr_state, curr_action, next_reward, next_state):
    old_value = q[(curr_state, curr_action)]
    future = 0.0 if next_state == simulator.terminal_state else max(
        q[(next_state, action)] for action in ACTIONS
    )
    new_value = old_value + ALPHA * (next_reward + GAMMA * future - old_value)
    q[(curr_state, curr_action)] = new_value
    return old_value, new_value

random.seed(2026)
q_random = make_q_table()
random_q_rows = []
for episode in range(32):
    for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(random_policy, max_steps=1024):
        old_value, new_value = q_update(q_random, curr_state, curr_action, next_reward, next_state)
        random_q_rows.append((curr_state, curr_action, next_reward, next_state, old_value, new_value))
        # Preserve the specified 1024 transitions per rollout for Part 2.
print("Random-policy updates:", len(random_q_rows))


Random-policy updates: 32768


Save each step in the simulator in a file "q-random.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

In [8]:
Q_COLUMNS = ["curr_state", "curr_action", "next_reward", "next_state", "old_value", "new_value"]
write_tsv("q-random.tsv", Q_COLUMNS, random_q_rows)


Submit "q-random.tsv" in Gradescope.

## Part 3: Implement Epsilon-Greedy Policy

Implement an epsilon-greedy policy that picks the optimal policy based on your q-values so far 75% of the time, and picks a random action 25% of the time.
This is a high epsilon value, but the environment is deterministic, so it will benefit from more exploration.

In [9]:
EPSILON = 0.25
q_greedy = make_q_table()

def best_action(q, state, actions=ACTIONS):
    # Random tie-breaking avoids systematic action-order bias during training.
    highest = max(q[(state, action)] for action in actions)
    choices = [action for action in actions if q[(state, action)] == highest]
    return random.choice(choices)

def epsilon_greedy_policy(state, actions):
    if random.random() < EPSILON:
        return random.choice(actions)
    return best_action(q_greedy, state, actions)


Combine your epsilon-greedy policy with q-learning below and save the observations and updates in "q-greedy.tsv" with columns curr_state, curr_action, next_reward, next_state, old_value, new_value.

Hint: make sure to reset your q-learning state before running the simulation below so that the learning process is recorded from the beginning.

In [11]:
random.seed(2026)
q_greedy = make_q_table()  # Reset before training, as required.
greedy_q_rows = []
for episode in range(32):
    for curr_state, curr_action, next_reward, next_state in simulator.rollout_policy(epsilon_greedy_policy, max_steps=1024):
        old_value, new_value = q_update(q_greedy, curr_state, curr_action, next_reward, next_state)
        greedy_q_rows.append((curr_state, curr_action, next_reward, next_state, old_value, new_value))
        if next_state == simulator.terminal_state:
            break
write_tsv("q-greedy.tsv", Q_COLUMNS, greedy_q_rows)
print("Epsilon-greedy updates:", len(greedy_q_rows))


Epsilon-greedy updates: 20679


Submit "q-greedy.tsv" in Gradescope.

## Part 4: Extract Policy from Q-Values

Using your final q-values from the previous simulation, extract a policy picking the best actions according to those q-values.
Save the policy in a file "policy-greedy.tsv" with columns state and action.

In [12]:
# Include all states so evaluation never encounters an undefined action.
ALL_STATES = range(simulator.resolution ** 2 * 8)
policy_greedy = [(state, best_action(q_greedy, state)) for state in ALL_STATES]
write_tsv("policy-greedy.tsv", ["state", "action"], policy_greedy)


Submit "policy-greedy.tsv" in Gradescope.

## Part 5: Implement Large Policy

Train a more optimal policy using q-learning.
Save the policy in a file "policy-optimal.tsv" with columns state and action.

Hint: this policy will be graded on its performance compared to optimal for each of the initial states.
**You will get full credit if the average value of your policy for the initial states is within 20% of optimal.**
Make sure that your policy has coverage of all the initial states, and does not take actions leading to states not included in your policy.
You will have to run several rollouts to get coverage of all the initial states, and the provided loops for parts 2 and 3 only consist of one rollout each.

Hint: this environment only gives one non-zero reward per episode, so you may want to cut off rollouts for speed once they get that reward.
But make sure you update the q-values first!

In [13]:
# Part 5: improve Q-learning with complete exploration and reverse Bellman backups.
# The deterministic environment permits exact model-based sweeps of all state-action
# transitions; these are Q-learning Bellman updates with alpha=1, gamma=0.9.
# Unlike random rollouts, sweeps guarantee every state/action receives updates.
q_optimal = make_q_table()
random.seed(2026)

# First, exploratory Q-learning from every possible nonterminal state.
# This also covers every required initial state.
for episode in range(12):
    starts = list(ALL_STATES)
    random.shuffle(starts)
    for start in starts:
        state = start
        for step in range(40):
            if state == simulator.terminal_state:
                break
            if random.random() < 0.30:
                action = random.choice(ACTIONS)
            else:
                action = best_action(q_optimal, state)
            reward, next_state = simulator.get_next_reward_state(state, action)
            q_update(q_optimal, state, action, reward, next_state)
            state = next_state

# Full-state Bellman sweeps propagate the sparse terminal reward to all states.
# This is particularly effective because transitions are deterministic.
for sweep in range(160):
    max_change = 0.0
    for state in ALL_STATES:
        if state == simulator.terminal_state:
            continue
        for action in ACTIONS:
            reward, next_state = simulator.get_next_reward_state(state, action)
            old_value, new_value = q_update(q_optimal, state, action, reward, next_state)
            max_change = max(max_change, abs(new_value - old_value))
    if max_change < 1e-12:
        print("Converged after", sweep + 1, "Bellman sweeps")
        break

# Deterministic tie-breaking for reproducible exported policy.
def extract_best(q, state):
    return max(ACTIONS, key=lambda action: (q[(state, action)], -abs(action), -action))

optimal_actions = {state: extract_best(q_optimal, state) for state in ALL_STATES}
write_tsv("policy-optimal.tsv", ["state", "action"], optimal_actions.items())

def evaluate_policy(actions_by_state, starts, max_steps=500):
    steps_taken = []
    for start in starts:
        state = start
        for step in range(max_steps):
            if state == simulator.terminal_state:
                steps_taken.append(step)
                break
            action = actions_by_state[state]
            _, state = simulator.get_next_reward_state(state, action)
        else:
            raise RuntimeError(f"Policy failed to reach terminal from state {start}")
    return sum(steps_taken) / len(steps_taken), max(steps_taken)

mean_steps, worst_steps = evaluate_policy(optimal_actions, simulator.initial_states)
print(f"Coverage: {len(optimal_actions)} states; initial states: {len(simulator.initial_states)}")
print(f"Mean steps to target: {mean_steps:.2f}; worst case: {worst_steps}")


Converged after 5 Bellman sweeps
Coverage: 2048 states; initial states: 71
Mean steps to target: 9.77; worst case: 13


Submit "policy-optimal.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [15]:
import pandas as pd 

acknowledgements = pd.DataFrame([
    {
        "source": "DX704 Week 8 Examples",
        "reference": "video_visualizing_pid_controllers.ipynb",
    }
])

acknowledgements.to_csv(
    "acknowledgements.txt",
    sep="\t",
    index=False
)